In [1]:
# =============================================================================
# CELL 1 — Imports & Configuration
# =============================================================================
# Purpose : Import all required libraries and define Gold layer configuration.
#
# Gold Layer Spark Optimizations applied:
#   - Window functions for lag features — fully distributed, no collect()
#   - broadcast() hint on holidays table — eliminates shuffle on large joins
#   - AQE (Adaptive Query Execution) — auto coalesces shuffle partitions
#   - Delta OPTIMIZE + ZORDER — improves file skipping for downstream reads
#   - cache() on base feature DataFrame — read multiple times across joins
#   - partitionBy(year, month, day) — partition pruning on Gold reads
#   - Native functions only — no UDFs, Catalyst-safe throughout
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, TimestampType, IntegerType, BooleanType
)
from delta.tables import DeltaTable

# -----------------------------------------------------------------------------
# Silver Source Paths
# -----------------------------------------------------------------------------
SILVER_PATHS = {
    "prices"    : "Tables/silver_electricity_prices",
    "load"      : "Tables/silver_electricity_load",
    "generation": "Tables/silver_generation_mix",
    "flows"     : "Tables/silver_cross_border_flows",
    "weather"   : "Tables/silver_weather"
}

# -----------------------------------------------------------------------------
# Gold Target Paths
# -----------------------------------------------------------------------------
GOLD_PATHS = {
    "price_features"   : "Tables/gold_price_features",
    "generation_summary" : "Tables/gold_generation_summary",
    "flow_summary"     : "Tables/gold_flow_summary",
    "price_aggregates" : "Tables/gold_price_aggregates"
}

# -----------------------------------------------------------------------------
# AQE — Adaptive Query Execution
# Enabled by default in Spark 3.x+ but explicitly confirmed here.
# AQE coalesces shuffle partitions post-aggregation — critical for Gold
# aggregations that produce small output from large shuffles.
# -----------------------------------------------------------------------------
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

print("✅ Imports and config loaded")
print(f"   Gold tables to build : {len(GOLD_PATHS)}")
print(f"   AQE enabled          : {spark.conf.get('spark.sql.adaptive.enabled')}")


StatementMeta(, e0a11b08-9dba-4612-83f3-b09dc3d14de9, 3, Finished, Available, Finished, False)

✅ Imports and config loaded
   Gold tables to build : 4
   AQE enabled          : true


In [2]:
# =============================================================================
# CELL 2 — Load Silver Tables
# =============================================================================
# Purpose : Read all 5 Silver Delta tables into Spark DataFrames.
#
# Spark Optimization — cache() on prices DataFrame:
#   - silver_prices is joined multiple times across Gold transformations
#     (lag features, rolling stats, aggregates)
#   - Caching avoids re-reading the Delta table on each join/transformation
#   - StorageLevel default (MEMORY_AND_DISK) — spills to disk if needed
#     on Trial cluster with limited memory
#
# Predicate pushdown on read:
#   - Delta file skipping applies automatically via column statistics
#   - Partitioned by year/month/day from Silver write — pruning applies
#     when Gold filters by date range
# =============================================================================

# Load all Silver tables
df_prices     = spark.read.format("delta").load(SILVER_PATHS["prices"])
df_load       = spark.read.format("delta").load(SILVER_PATHS["load"])
df_generation = spark.read.format("delta").load(SILVER_PATHS["generation"])
df_flows      = spark.read.format("delta").load(SILVER_PATHS["flows"])
df_weather    = spark.read.format("delta").load(SILVER_PATHS["weather"])

# Cache prices — used multiple times across Gold feature engineering
# Avoids re-reading Delta on each transformation
df_prices.cache()

print("✅ Silver tables loaded")
print(f"   prices     : {df_prices.count()} rows (cached)")
print(f"   load       : {df_load.count()} rows")
print(f"   generation : {df_generation.count()} rows")
print(f"   flows      : {df_flows.count()} rows")
print(f"   weather    : {df_weather.count()} rows")

StatementMeta(, e0a11b08-9dba-4612-83f3-b09dc3d14de9, 4, Finished, Available, Finished, False)

✅ Silver tables loaded
   prices     : 66 rows (cached)
   load       : 140 rows
   generation : 1689 rows
   flows      : 105 rows
   weather    : 38 rows


In [3]:
# =============================================================================
# CELL 3 — Holidays Reference Table
# =============================================================================
# Purpose : Create a small holidays lookup table for the is_holiday feature.
#           Used in Gold feature engineering as a broadcast join.
#
# Spark Optimization — broadcast() hint:
#   - Holidays table is ~20 rows — tiny reference table
#   - Without broadcast: Spark shuffles the large prices table to join
#   - With broadcast: Spark sends the small holidays table to every executor
#   - Eliminates the shuffle entirely on the larger prices table
#   - Rule of thumb: broadcast tables < 10MB (this is ~1KB)
# =============================================================================

from pyspark.sql import Row
from datetime import date

# European + US public holidays for 2026 (key market dates)
holidays_data = [
    Row(holiday_date="2026-01-01", holiday_name="New Year"),
    Row(holiday_date="2026-01-06", holiday_name="Epiphany"),
    Row(holiday_date="2026-04-03", holiday_name="Good Friday"),
    Row(holiday_date="2026-04-06", holiday_name="Easter Monday"),
    Row(holiday_date="2026-05-01", holiday_name="Labour Day"),
    Row(holiday_date="2026-05-14", holiday_name="Ascension Day"),
    Row(holiday_date="2026-05-25", holiday_name="Whit Monday"),
    Row(holiday_date="2026-07-04", holiday_name="US Independence Day"),
    Row(holiday_date="2026-08-15", holiday_name="Assumption Day"),
    Row(holiday_date="2026-10-03", holiday_name="German Unity Day"),
    Row(holiday_date="2026-11-01", holiday_name="All Saints Day"),
    Row(holiday_date="2026-11-26", holiday_name="US Thanksgiving"),
    Row(holiday_date="2026-12-24", holiday_name="Christmas Eve"),
    Row(holiday_date="2026-12-25", holiday_name="Christmas Day"),
    Row(holiday_date="2026-12-26", holiday_name="Boxing Day"),
    Row(holiday_date="2026-12-31", holiday_name="New Year Eve"),
]

df_holidays = spark.createDataFrame(holidays_data) \
    .withColumn("holiday_date", F.to_date(F.col("holiday_date")))

# Explicit broadcast hint — eliminates shuffle on join with prices table
df_holidays_broadcast = F.broadcast(df_holidays)

print(f"✅ Holidays table created: {df_holidays.count()} entries")
print("   Broadcast hint applied — shuffle eliminated on join")

StatementMeta(, e0a11b08-9dba-4612-83f3-b09dc3d14de9, 5, Finished, Available, Finished, False)

✅ Holidays table created: 16 entries
   Broadcast hint applied — shuffle eliminated on join


In [4]:
# =============================================================================
# CELL 4 — Gold Table 1: gold_price_features (Core ML Feature Table)
# =============================================================================
# Purpose : Engineer features for the XGBoost price spike predictor.
#           This is the primary input table for Phase 4 ML training.
#
# Spark Optimizations:
#   - Window functions for lag/rolling features — fully distributed
#     No collect() to driver — all computation stays in Spark executors
#   - broadcast() on holidays — eliminates shuffle (applied in Cell 3)
#   - Native functions only — F.lag, F.avg, F.stddev, F.percentile_approx
#     all Catalyst-visible
#   - drop() duplicate columns before joins — prevents AnalysisException
#     on columns that exist in both prices (from Bronze) and Silver tables
#
# Features engineered:
#   - price_lag_1h          : Price 1 hour ago (short-term momentum)
#   - price_lag_12h         : Price 12 hours ago (half-day pattern)
#   - price_lag_24h         : Price 24 hours ago (same-hour yesterday)
#   - price_rolling_avg_6h  : 6-hour rolling mean (trend)
#   - price_rolling_std_6h  : 6-hour rolling std dev (volatility)
#   - hour_of_day           : 0-23 (peak hour indicator)
#   - day_of_week           : 1-7 (weekday vs weekend)
#   - is_weekend            : Boolean (lower industrial demand)
#   - is_holiday            : Boolean (demand profile shifts)
#   - temperature_c         : From weather table (heating/cooling demand)
#   - wind_speed_ms         : Wind generation proxy (suppresses prices)
#   - humidity_pct          : Weather enrichment
#   - solar_radiation       : Solar generation proxy (suppresses prices)
#   - load_mw               : From load table (15-min granularity)
#   - is_spike              : Target label (price > 90th percentile)
# =============================================================================

# -----------------------------------------------------------------------------
# Step 1 — Window spec for lag + rolling features
# Partition by region, order by event_time
# All computation fully distributed in Spark executors
# -----------------------------------------------------------------------------
window_lag = Window \
    .partitionBy("region") \
    .orderBy("event_time")

window_rolling_6h = Window \
    .partitionBy("region") \
    .orderBy(F.col("event_time").cast("long")) \
    .rangeBetween(-6 * 3600, 0)   # 6-hour rolling window in seconds

# -----------------------------------------------------------------------------
# Step 2 — Compute price percentile threshold per region
# Used to define is_spike target label
# percentile_approx is Catalyst-optimized — no UDF needed
# -----------------------------------------------------------------------------
df_percentile = df_prices \
    .groupBy("region") \
    .agg(
        F.percentile_approx("price_eur_mwh", 0.9).alias("p90_price")
    )

# -----------------------------------------------------------------------------
# Step 3 — Engineer lag + rolling features using window functions
# -----------------------------------------------------------------------------
df_lag_features = df_prices \
    .withColumn("price_lag_1h",
        F.lag("price_eur_mwh", 1).over(window_lag)
    ) \
    .withColumn("price_lag_12h",
        F.lag("price_eur_mwh", 12).over(window_lag)
    ) \
    .withColumn("price_lag_24h",
        F.lag("price_eur_mwh", 24).over(window_lag)
    ) \
    .withColumn("price_rolling_avg_6h",
        F.avg("price_eur_mwh").over(window_rolling_6h)
    ) \
    .withColumn("price_rolling_std_6h",
        F.stddev("price_eur_mwh").over(window_rolling_6h)
    ) \
    .withColumn("hour_of_day",   F.hour("event_time")) \
    .withColumn("day_of_week",   F.dayofweek("event_time")) \
    .withColumn("is_weekend",
        F.when(F.dayofweek("event_time").isin([1, 7]), True)
        .otherwise(False)
    ) \
    .withColumn("event_date", F.to_date("event_time"))

# -----------------------------------------------------------------------------
# Step 4 — Join holidays (broadcast — no shuffle on prices table)
# -----------------------------------------------------------------------------
df_with_holidays = df_lag_features \
    .join(
        df_holidays_broadcast,
        df_lag_features["event_date"] == df_holidays["holiday_date"],
        how="left"
    ) \
    .withColumn("is_holiday",
        F.when(F.col("holiday_name").isNotNull(), True).otherwise(False)
    ) \
    .drop("holiday_date", "holiday_name", "event_date")

# -----------------------------------------------------------------------------
# Step 5 — Join weather features
# Drop temperature_c from prices first — already exists from Bronze schema
# Weather table provides enriched version with wind, humidity, solar
# -----------------------------------------------------------------------------
df_with_weather = df_with_holidays \
    .drop("temperature_c") \
    .join(
        df_weather.select(
            "region", "event_time",
            "temperature_c", "wind_speed_ms",
            "humidity_pct", "solar_radiation"
        ),
        on=["region", "event_time"],
        how="left"
    )

# -----------------------------------------------------------------------------
# Step 6 — Join load features
# Drop load_mw from prices first — already exists from Bronze schema
# Load table provides dedicated 15-min granularity version
# -----------------------------------------------------------------------------
df_with_load = df_with_weather \
    .drop("load_mw") \
    .join(
        df_load.select("region", "event_time", "load_mw"),
        on=["region", "event_time"],
        how="left"
    )

# -----------------------------------------------------------------------------
# Step 7 — Join p90 threshold + compute is_spike target label
# -----------------------------------------------------------------------------
df_gold_features = df_with_load \
    .join(df_percentile, on="region", how="left") \
    .withColumn("is_spike",
        F.when(F.col("price_eur_mwh") > F.col("p90_price"), True)
        .otherwise(False)
    ) \
    .drop("p90_price") \
    .withColumn("year",  F.year("event_time")) \
    .withColumn("month", F.month("event_time")) \
    .withColumn("day",   F.dayofmonth("event_time"))

# -----------------------------------------------------------------------------
# Step 8 — Write to Gold Delta
# -----------------------------------------------------------------------------
gold_path = GOLD_PATHS["price_features"]

if DeltaTable.isDeltaTable(spark, gold_path):
    DeltaTable.forPath(spark, gold_path).alias("t").merge(
        df_gold_features.alias("s"),
        "t.region = s.region AND t.event_time = s.event_time"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print("✅ gold_price_features — MERGE complete")
else:
    df_gold_features.write \
        .format("delta") \
        .mode("overwrite") \
        .partitionBy("year", "month", "day") \
        .save(gold_path)
    print("✅ gold_price_features — created")

print(f"   Rows written : {df_gold_features.count()}")
print(f"   Spikes (1)   : {df_gold_features.filter(F.col('is_spike') == True).count()}")
print(f"   Non-spike (0): {df_gold_features.filter(F.col('is_spike') == False).count()}")
print(f"   Features     : {len(df_gold_features.columns)} columns")

StatementMeta(, e0a11b08-9dba-4612-83f3-b09dc3d14de9, 6, Finished, Available, Finished, False)

✅ gold_price_features — MERGE complete
   Rows written : 66
   Spikes (1)   : 0
   Non-spike (0): 66
   Features     : 23 columns


In [5]:
# =============================================================================
# CELL 5 — Gold Table 2: gold_generation_summary
# =============================================================================
# Purpose : Compute hourly generation mix ratios per region.
#           Renewable % (solar + wind) is a strong negative price predictor.
#           Nuclear % is a strong price stability indicator.
#
# Spark Optimization — AQE:
#   - groupBy + pivot creates a wide shuffle
#   - AQE coalesces the shuffle partitions automatically post-aggregation
#   - No manual repartition needed here — AQE handles it
#
# Output columns (one per fuel type + computed ratios):
#   - solar_mw, wind_onshore_mw, nuclear_mw, gas_mw, hydro_mw
#   - total_generation_mw
#   - renewable_pct  : (solar + wind) / total × 100
#   - nuclear_pct    : nuclear / total × 100
#   - fossil_pct     : gas / total × 100
# =============================================================================

# Pivot fuel types to columns — one row per region per hour
df_gen_pivot = df_generation \
    .withColumn("hour_time",
        F.date_trunc("hour", F.col("event_time"))
    ) \
    .groupBy("region", "hour_time") \
    .pivot("fuel_type", [
        "Solar", "Wind Onshore", "Wind Offshore",
        "Nuclear", "Gas", "Hydro", "Coal", "Other"
    ]) \
    .agg(F.sum("generation_mw")) \
    .fillna(0.0)

# Rename pivoted columns — replace spaces with underscores, lowercase
for fuel in ["Solar", "Wind Onshore", "Wind Offshore",
             "Nuclear", "Gas", "Hydro", "Coal", "Other"]:
    safe_name = fuel.lower().replace(" ", "_") + "_mw"
    if fuel in df_gen_pivot.columns:
        df_gen_pivot = df_gen_pivot.withColumnRenamed(fuel, safe_name)

# Compute generation ratios
df_gen_summary = df_gen_pivot \
    .withColumn("total_generation_mw",
        F.coalesce(F.col("solar_mw"), F.lit(0.0)) +
        F.coalesce(F.col("wind_onshore_mw"), F.lit(0.0)) +
        F.coalesce(F.col("nuclear_mw"), F.lit(0.0)) +
        F.coalesce(F.col("gas_mw"), F.lit(0.0)) +
        F.coalesce(F.col("hydro_mw"), F.lit(0.0))
    ) \
    .withColumn("renewable_pct",
        F.when(F.col("total_generation_mw") > 0,
            (F.coalesce(F.col("solar_mw"), F.lit(0.0)) +
             F.coalesce(F.col("wind_onshore_mw"), F.lit(0.0))) /
            F.col("total_generation_mw") * 100
        ).otherwise(0.0)
    ) \
    .withColumn("nuclear_pct",
        F.when(F.col("total_generation_mw") > 0,
            F.coalesce(F.col("nuclear_mw"), F.lit(0.0)) /
            F.col("total_generation_mw") * 100
        ).otherwise(0.0)
    ) \
    .withColumn("fossil_pct",
        F.when(F.col("total_generation_mw") > 0,
            F.coalesce(F.col("gas_mw"), F.lit(0.0)) /
            F.col("total_generation_mw") * 100
        ).otherwise(0.0)
    ) \
    .withColumnRenamed("hour_time", "event_time") \
    .withColumn("year",  F.year("event_time")) \
    .withColumn("month", F.month("event_time")) \
    .withColumn("day",   F.dayofmonth("event_time"))

gold_path = GOLD_PATHS["generation_summary"]
if DeltaTable.isDeltaTable(spark, gold_path):
    DeltaTable.forPath(spark, gold_path).alias("t").merge(
        df_gen_summary.alias("s"),
        "t.region = s.region AND t.event_time = s.event_time"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print("✅ gold_generation_summary — MERGE complete")
else:
    df_gen_summary.write \
        .format("delta") \
        .mode("overwrite") \
        .partitionBy("year", "month", "day") \
        .save(gold_path)
    print("✅ gold_generation_summary — created")

print(f"   Rows written : {df_gen_summary.count()}")

StatementMeta(, e0a11b08-9dba-4612-83f3-b09dc3d14de9, 7, Finished, Available, Finished, False)

✅ gold_generation_summary — MERGE complete
   Rows written : 58


In [6]:
# =============================================================================
# CELL 6 — Gold Table 3: gold_flow_summary
# =============================================================================
# Purpose : Compute net cross-border flow position per region per hour.
#           Net importer regions tend to have higher prices.
#           Net exporter regions tend to have lower prices.
#
# Logic:
#   - Export flow  : from_region = this region → positive MW leaving
#   - Import flow  : to_region   = this region → positive MW arriving
#   - Net position : total_exports - total_imports
#                    Positive = net exporter, Negative = net importer
# =============================================================================

# Exports — flows leaving the region
df_exports = df_flows \
    .withColumn("hour_time", F.date_trunc("hour", F.col("event_time"))) \
    .groupBy(F.col("from_region").alias("region"), "hour_time") \
    .agg(F.sum("flow_mw").alias("total_exports_mw"))

# Imports — flows arriving at the region
df_imports = df_flows \
    .withColumn("hour_time", F.date_trunc("hour", F.col("event_time"))) \
    .groupBy(F.col("to_region").alias("region"), "hour_time") \
    .agg(F.sum(F.abs("flow_mw")).alias("total_imports_mw"))

# Join exports + imports → net position
df_flow_summary = df_exports \
    .join(df_imports, on=["region", "hour_time"], how="outer") \
    .fillna(0.0) \
    .withColumn("net_flow_mw",
        F.col("total_exports_mw") - F.col("total_imports_mw")
    ) \
    .withColumn("flow_position",
        F.when(F.col("net_flow_mw") > 0, "Exporter")
        .when(F.col("net_flow_mw") < 0, "Importer")
        .otherwise("Balanced")
    ) \
    .withColumnRenamed("hour_time", "event_time") \
    .withColumn("year",  F.year("event_time")) \
    .withColumn("month", F.month("event_time")) \
    .withColumn("day",   F.dayofmonth("event_time"))

gold_path = GOLD_PATHS["flow_summary"]
if DeltaTable.isDeltaTable(spark, gold_path):
    DeltaTable.forPath(spark, gold_path).alias("t").merge(
        df_flow_summary.alias("s"),
        "t.region = s.region AND t.event_time = s.event_time"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print("✅ gold_flow_summary — MERGE complete")
else:
    df_flow_summary.write \
        .format("delta") \
        .mode("overwrite") \
        .partitionBy("year", "month", "day") \
        .save(gold_path)
    print("✅ gold_flow_summary — created")

print(f"   Rows written : {df_flow_summary.count()}")

StatementMeta(, e0a11b08-9dba-4612-83f3-b09dc3d14de9, 8, Finished, Available, Finished, False)

✅ gold_flow_summary — MERGE complete
   Rows written : 44


In [7]:
# =============================================================================
# CELL 7 — Gold Table 4: gold_price_aggregates (Power BI Layer)
# =============================================================================
# Purpose : Compute hourly and daily price aggregates per region.
#           Primary source for the Power BI Semantic Model and dashboard.
#
# Spark Optimization — AQE on groupBy:
#   - Two groupBy operations (hourly + daily) each trigger shuffles
#   - AQE coalesces output partitions automatically post-aggregation
#   - No manual repartition needed — AQE handles it
#   - union() combines hourly + daily into one table — single Delta write
#
# Fix — Duplicate column handling:
#   - df_prices contains load_mw from Bronze schema
#   - df_load also contains load_mw
#   - Drop load_mw from df_prices before join to avoid ambiguous reference
#
# Metrics computed:
#   - avg_price, min_price, max_price, price_range
#   - record_count : number of ticks in the period
#   - avg_load     : average grid load
#   - avg_temp     : average temperature
# =============================================================================

# Drop duplicate columns from prices before enrichment joins
df_prices_clean = df_prices \
    .drop("load_mw", "temperature_c")

# Join prices with load + weather for enriched aggregates
df_enriched = df_prices_clean \
    .join(
        df_load.select("region", "event_time", "load_mw"),
        on=["region", "event_time"],
        how="left"
    ) \
    .join(
        df_weather.select("region", "event_time", "temperature_c"),
        on=["region", "event_time"],
        how="left"
    )

# Hourly aggregates
df_hourly = df_enriched \
    .withColumn("period_start", F.date_trunc("hour", F.col("event_time"))) \
    .groupBy("region", "period_start") \
    .agg(
        F.avg("price_eur_mwh").alias("avg_price"),
        F.min("price_eur_mwh").alias("min_price"),
        F.max("price_eur_mwh").alias("max_price"),
        (F.max("price_eur_mwh") - F.min("price_eur_mwh")).alias("price_range"),
        F.count("price_eur_mwh").alias("record_count"),
        F.avg("load_mw").alias("avg_load"),
        F.avg("temperature_c").alias("avg_temp")
    ) \
    .withColumn("granularity", F.lit("hourly"))

# Daily aggregates
df_daily = df_enriched \
    .withColumn("period_start", F.to_timestamp(F.to_date("event_time"))) \
    .groupBy("region", "period_start") \
    .agg(
        F.avg("price_eur_mwh").alias("avg_price"),
        F.min("price_eur_mwh").alias("min_price"),
        F.max("price_eur_mwh").alias("max_price"),
        (F.max("price_eur_mwh") - F.min("price_eur_mwh")).alias("price_range"),
        F.count("price_eur_mwh").alias("record_count"),
        F.avg("load_mw").alias("avg_load"),
        F.avg("temperature_c").alias("avg_temp")
    ) \
    .withColumn("granularity", F.lit("daily"))

# Union hourly + daily into one Gold table
df_price_aggregates = df_hourly.union(df_daily) \
    .withColumn("year",  F.year("period_start")) \
    .withColumn("month", F.month("period_start")) \
    .withColumn("day",   F.dayofmonth("period_start"))

gold_path = GOLD_PATHS["price_aggregates"]
if DeltaTable.isDeltaTable(spark, gold_path):
    DeltaTable.forPath(spark, gold_path).alias("t").merge(
        df_price_aggregates.alias("s"),
        "t.region = s.region AND t.period_start = s.period_start AND t.granularity = s.granularity"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print("✅ gold_price_aggregates — MERGE complete")
else:
    df_price_aggregates.write \
        .format("delta") \
        .mode("overwrite") \
        .partitionBy("year", "month", "day") \
        .save(gold_path)
    print("✅ gold_price_aggregates — created")

print(f"   Rows written : {df_price_aggregates.count()}")
print(f"   Hourly rows  : {df_hourly.count()}")
print(f"   Daily rows   : {df_daily.count()}")

StatementMeta(, e0a11b08-9dba-4612-83f3-b09dc3d14de9, 9, Finished, Available, Finished, False)

✅ gold_price_aggregates — MERGE complete
   Rows written : 84
   Hourly rows  : 66
   Daily rows   : 18


In [8]:
# =============================================================================
# CELL 8 — Delta OPTIMIZE + ZORDER
# =============================================================================
# Purpose : Optimize all Gold Delta tables for downstream read performance.
#
# OPTIMIZE:
#   - Compacts small Delta files into larger ones (target ~128MB)
#   - Reduces the number of files Spark/Power BI must open per query
#   - Essential after incremental MERGE writes which produce many small files
#
# ZORDER BY (region, event_time):
#   - Co-locates rows with the same region + event_time in the same files
#   - Enables file skipping: queries filtered by region skip irrelevant files
#   - Power BI Semantic Model and Streamlit agent both filter by region
#   - Most impactful optimization for query performance on Gold tables
#
# When to run:
#   - After bulk loads or MERGE operations
#   - Scheduled weekly via Fabric Pipeline (not on every notebook run)
#   - Here we run once after initial Gold table creation
# =============================================================================

optimize_targets = [
    (GOLD_PATHS["price_features"],    ["region", "event_time"]),
    (GOLD_PATHS["generation_summary"],["region", "event_time"]),
    (GOLD_PATHS["flow_summary"],      ["region", "event_time"]),
    (GOLD_PATHS["price_aggregates"],  ["region", "period_start"]),
]

for path, zorder_cols in optimize_targets:
    table_name = path.replace("Tables/", "")
    zorder_str = ", ".join(zorder_cols)
    try:
        spark.sql(f"OPTIMIZE delta.`{path}` ZORDER BY ({zorder_str})")
        print(f"✅ OPTIMIZE + ZORDER complete : {table_name}")
    except Exception as e:
        print(f"⚠️  OPTIMIZE skipped for {table_name}: {e}")

StatementMeta(, e0a11b08-9dba-4612-83f3-b09dc3d14de9, 10, Finished, Available, Finished, False)

✅ OPTIMIZE + ZORDER complete : gold_price_features
✅ OPTIMIZE + ZORDER complete : gold_generation_summary
✅ OPTIMIZE + ZORDER complete : gold_flow_summary
✅ OPTIMIZE + ZORDER complete : gold_price_aggregates


In [9]:
# =============================================================================
# CELL 9 — Gold Layer Validation
# =============================================================================
# Purpose : Validate all 4 Gold tables — row counts, spike distribution,
#           feature coverage, and date range.
# =============================================================================

print("=" * 55)
print("  Gold Layer — Validation Report")
print("=" * 55)

# gold_price_features
df_gf = spark.read.format("delta").load(GOLD_PATHS["price_features"])
spike_count    = df_gf.filter(F.col("is_spike") == True).count()
non_spike      = df_gf.filter(F.col("is_spike") == False).count()
lag_nulls      = df_gf.filter(F.col("price_lag_1h").isNull()).count()
date_range     = df_gf.agg(F.min("event_time"), F.max("event_time")).collect()[0]

print(f"\n  ✅ gold_price_features")
print(f"     Total rows     : {df_gf.count()}")
print(f"     Spikes (1)     : {spike_count}")
print(f"     Non-spike (0)  : {non_spike}")
print(f"     Lag nulls      : {lag_nulls} (expected for first records)")
print(f"     Date range     : {date_range[0]} → {date_range[1]}")
print(f"     Features       : {len(df_gf.columns)} columns")

# gold_generation_summary
df_gs = spark.read.format("delta").load(GOLD_PATHS["generation_summary"])
print(f"\n  ✅ gold_generation_summary")
print(f"     Total rows     : {df_gs.count()}")
print(f"     Regions        : {df_gs.select('region').distinct().count()}")
df_gs.select("region", "renewable_pct", "nuclear_pct", "fossil_pct").show(5, truncate=False)

# gold_flow_summary
df_fs = spark.read.format("delta").load(GOLD_PATHS["flow_summary"])
print(f"\n  ✅ gold_flow_summary")
print(f"     Total rows     : {df_fs.count()}")
df_fs.select("region", "net_flow_mw", "flow_position").show(5, truncate=False)

# gold_price_aggregates
df_pa = spark.read.format("delta").load(GOLD_PATHS["price_aggregates"])
hourly = df_pa.filter(F.col("granularity") == "hourly").count()
daily  = df_pa.filter(F.col("granularity") == "daily").count()
print(f"\n  ✅ gold_price_aggregates")
print(f"     Total rows     : {df_pa.count()}")
print(f"     Hourly records : {hourly}")
print(f"     Daily records  : {daily}")
df_pa.select("region", "period_start", "avg_price", "granularity").show(5, truncate=False)

print("\n" + "=" * 55)
print("  Gold Layer — Complete ✅")
print("=" * 55)

StatementMeta(, e0a11b08-9dba-4612-83f3-b09dc3d14de9, 11, Finished, Available, Finished, False)

  Gold Layer — Validation Report

  ✅ gold_price_features
     Total rows     : 66
     Spikes (1)     : 0
     Non-spike (0)  : 66
     Lag nulls      : 41 (expected for first records)
     Date range     : 2026-08-13 06:00:00 → 2026-08-14 07:00:00
     Features       : 23 columns

  ✅ gold_generation_summary
     Total rows     : 58
     Regions        : 27
+------+------------------+-----------------+----------+
|region|renewable_pct     |nuclear_pct      |fossil_pct|
+------+------------------+-----------------+----------+
|AT    |0.0               |0.0              |0.0       |
|HU    |0.0               |0.0              |0.0       |
|EE    |100.0             |0.0              |0.0       |
|GR    |0.0               |0.0              |0.0       |
|CH    |1.8401795507873502|98.15982044921266|0.0       |
+------+------------------+-----------------+----------+
only showing top 5 rows


  ✅ gold_flow_summary
     Total rows     : 44
+------+-----------+-------------+
|region|net_flow_